### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import torch, os

# Adjust to point to the actual root of your project
PROJECT_ROOT = Path.cwd().parent  # or Path("/absolute/path/to/your/project")
sys.path.insert(0, str(PROJECT_ROOT))

### Loading the model

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

load_model = False

model_name = "deepseek-ai/DeepSeek-V2-Lite"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if load_model:
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map='auto', 
        attn_implementation='eager',  
        trust_remote_code=True
    )
    model.generation_config = GenerationConfig.from_pretrained(model_name)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
    model.eval()

    text = "The goal of life is to"
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs.to(model.device), max_new_tokens=10)

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(result)

### Record activations, save them, and verify them

In [5]:
from src.activation_recorder import ActivationRecorder, MultiPromptActivations
from IPython.core.debugger import Pdb

load_from_disk = True
save_dir = "../data/activations"
max_new_tokens=10
prompts = ["Tell me a joke", "Hello, world! How can you code"]

if not load_from_disk:
    recorder = ActivationRecorder(model, tokenizer)
    activations = recorder.record_prompts(prompts, max_new_tokens=max_new_tokens)
    activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)
    activations.save(save_dir)

# Load the activations from disk.
file_path = os.path.join(save_dir, "multi_prompt_activations.pkl")
loaded_activations = MultiPromptActivations.load(file_path)

# Optional: verify the loaded activations match the saved ones.
print("Loaded MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Check again the activations
loaded_activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)

print("Final MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Extract the first prompt, first step, first layer, first head
prompt_acts = loaded_activations.prompts[0]
step_acts = prompt_acts.steps[0]
layer_acts = step_acts.layers[0]
attn = layer_acts.attention
for head_idx, head_acts in attn.heads.items():
    print(f"Head {head_idx} activations:")
    print(head_acts.query.shape)
    print(head_acts.attention_weights.shape)
    print(head_acts.attention_outputs.shape)
    print(head_acts.projected_outputs.shape)

MultiPromptActivations successfully loaded from '../data/activations/multi_prompt_activations.pkl'.
Loaded MultiPromptActivations object has: 2 prompts recorded.
Activations check passed!
Final MultiPromptActivations object has: 2 prompts recorded.
Head 0 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 1 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 2 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 3 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 4 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 5 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 6 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 7 activations:
torch.Size([192])
torch.Size([5])
torch.Size([128])
torch.Size([2048])
Head 8 activations:
tor

In [ ]:
step_acts = prompt_acts.steps[4]
for layer_idx, layer_acts in step_acts.layers.items():
    print(f"Layer {layer_idx} activations:")
    moe_layer_acts = step_acts.layers[2].moe
    for expert_id, expert_acts in moe_layer_acts.experts.items():
        print(f"Expert {expert_id} activations:")
        print(repr(expert_acts))
        print("Gate value:", expert_acts.gate_value)
        print("MLP output:", expert_acts.mlp_output)
        print("Expert output:", expert_acts.expert_output)

Layer 0 activations:
Expert 0 activations:
MoEExpertActivations(layer_index=2, expert_index=0, gate_value=torch.Size([]), mlp_output=torch.Size([2048]), expert_output=torch.Size([2048]), is_shared=False, model_info=ModelInformation(model_name=deepseek-ai/DeepSeek-V2-Lite, model_architecture=DeepseekV2ForCausalLM, num_layers=27, num_attention_heads_per_layer=16, total_num_attention_heads=432, attention_implementation=default, hidden_size=2048, head_dim=128, attention_implementation=default))
Gate value: tensor(0.0571, device='cuda:0')
MLP output: tensor([ 1.3534e-04,  3.2173e-05, -3.3733e-04,  ..., -8.0709e-04,
        -5.2958e-05,  6.5426e-04], device='cuda:0')
Expert output: tensor([ 1.3534e-04,  3.2173e-05, -3.3733e-04,  ..., -8.0709e-04,
        -5.2958e-05,  6.5426e-04], device='cuda:0')
Expert 1 activations:
MoEExpertActivations(layer_index=2, expert_index=1, gate_value=torch.Size([]), mlp_output=torch.Size([2048]), expert_output=torch.Size([2048]), is_shared=False, model_info=Mod